# TrailForge — DKT Training Notebook

**What this notebook does:**
1. Generates a synthetic student dataset
2. Trains one DKT model per topic
3. Downloads the trained `.pt` files to your laptop

**You paste those `.pt` files into your FastAPI backend. That's it.**

---
### Before you start
Make sure GPU is ON:
`Runtime → Change runtime type → T4 GPU → Save`

## Cell 1 — Check GPU

In [ ]:
import torch
print('PyTorch version :', torch.__version__)
print('GPU available   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name        :', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU')

## Cell 2 — Install dependencies

In [ ]:
!pip install torch scikit-learn pandas numpy matplotlib --quiet
print('Done.')

## Cell 3 — Upload your 5 Python files

Upload these files from the `colab/` folder on your laptop:
- `skills.py`
- `simulate.py`
- `encode.py`
- `model.py`
- `train.py`

In [ ]:
from google.colab import files
print('Select all 5 .py files at once and upload:')
uploaded = files.upload()
print('\nUploaded files:', list(uploaded.keys()))

# Verify all files are there
required = ['skills.py', 'simulate.py', 'encode.py', 'model.py', 'train.py']
missing  = [f for f in required if f not in uploaded]
if missing:
    print(f'MISSING: {missing} — please upload these too')
else:
    print('All files uploaded correctly ✓')

## Cell 4 — Check topic and concept definitions

In [ ]:
from skills import TOPICS, DIFFICULTIES

print('Topics and concepts loaded:')
for topic, concepts in TOPICS.items():
    print(f'  {topic} ({len(concepts)} concepts): {concepts}')
print(f'\nDifficulties: {DIFFICULTIES}')

## Cell 5 — Generate dataset

Simulates 2000 virtual students.
Takes about 1-2 minutes.
Produces `dkt_dataset.csv`.

In [ ]:
from simulate import generate_dataset, print_stats
import pandas as pd

df = generate_dataset()
print_stats(df)
df.to_csv('dkt_dataset.csv', index=False)
print('\nSaved: dkt_dataset.csv')

## Cell 6 — Sanity check the dataset

In [ ]:
df = pd.read_csv('dkt_dataset.csv')

print('First 15 rows of student 0 on java:')
sample = df[(df['student_id']==0) & (df['topic']=='java')].head(15)
print(sample.to_string(index=False))

print('\nCorrect rate by difficulty:')
print(df.groupby('difficulty')['correct'].mean().round(3))

print('\nInteraction count by topic:')
print(df.groupby('topic').size())

## Cell 7 — Train models

Trains one model per topic.
With T4 GPU takes about 5-8 minutes total.
Produces `dkt_model_java.pt`, `dkt_model_python.pt`, `dkt_model_sql.pt`.

In [ ]:
from train import train
train()

## Cell 8 — Quick inference test (confirm model works)

In [ ]:
import torch
from model  import DKT
from encode import encode_interaction, input_size
from skills import get_concepts, get_concept_to_idx, num_concepts, DIFFICULTIES, NUM_DIFFICULTIES
import numpy as np

topic = 'java'

# Load model
ckpt  = torch.load(f'dkt_model_{topic}.pt', map_location='cpu')
cfg   = ckpt['config']
model = DKT(topic=topic, hidden_size=cfg['hidden_size'],
            num_layers=cfg['num_layers'], dropout=cfg['dropout'])
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Model loaded: val_loss={ckpt["val_loss"]:.6f}  epoch={ckpt["epoch"]}')

# Test interaction sequence
concepts = get_concepts(topic)
c2i      = get_concept_to_idx(topic)
n_c      = num_concepts(topic)

history = [
    ('loops', 'beginner',     1),
    ('loops', 'beginner',     1),
    ('loops', 'intermediate', 0),
    ('loops', 'intermediate', 1),
    ('loops', 'intermediate', 1),
]

encoded = np.array(
    [encode_interaction(c, d, cor, c2i, n_c) for c, d, cor in history],
    dtype=np.float32
)
x = torch.tensor(encoded).unsqueeze(0)

with torch.no_grad():
    output = model(x)

last = output[0, -1, :].numpy().reshape(n_c, NUM_DIFFICULTIES)

print('\nMastery after 5 interactions on loops:')
for c_idx, concept in enumerate(concepts):
    beg = last[c_idx, 0]
    mid = last[c_idx, 1]
    adv = last[c_idx, 2]
    tag = ' ← trained' if concept == 'loops' else ''
    print(f'  {concept:<18} beg={beg:.2f}  int={mid:.2f}  adv={adv:.2f}{tag}')

## Cell 9 — Download model files

**Download all `.pt` files to your laptop.**
Then paste them into `backend/dkt/weights/` in your project.

In [ ]:
import os
from google.colab import files
from skills import TOPICS

to_download = [f'dkt_model_{topic}.pt' for topic in TOPICS]
to_download += ['training_log.json']

print('Downloading files to your laptop ...')
for fname in to_download:
    if os.path.exists(fname):
        files.download(fname)
        print(f'  [↓] {fname}')
    else:
        print(f'  [!] {fname} not found — did training finish?')

print('\nDone. Now paste the .pt files into:')
print('  your_project/backend/dkt/weights/')